# Подбор весов поиска по корпусу

Поиск при обучении (`presentation_rl.tools.search`, BM25F по разделам) должен выдавать примерно то же, что живой
поиск Википедии, который модель увидит при использовании. Сравниваем на пилотном корпусе (`data/sources/wiki_pilot/`):
на каждый запрос из `topics_v2.jsonl` есть живая выдача; берём из неё статьи, которые есть в корпусе (топ-3),
и смотрим, сколько из них попало в наш топ-3 и совпало ли первое место.

Ищем по всему корпусу сразу (все 8 тем), как и при обучении. Раздел внутри статьи в обоих режимах выбирает наш BM25,
поэтому сравниваем только статьи.

Результат 16.10.2026: `title=20, heading=3, article=0.3, k1=1.2, b=0.75` — совпадение топ-3 0.66, первое место 0.59;
без весов полей (чистый BM25 по тексту раздела) — 0.56 и 0.41. Чего у нас нет: перенаправлений (сокращения вроде
«НАДФН», «ИПЦ») и популярности статей.

In [ ]:
import itertools, json, random, sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from presentation_rl.tools.search import SectionIndex, Weights
from presentation_rl.tools.wiki import Article

DATA = ROOT / "data" / "sources" / "wiki_pilot"
articles = [Article.from_json(r) for r in map(json.loads, (DATA / "articles.jsonl").open(encoding="utf-8"))
            if not r.get("missing") and r["lang"] == "ru"]
index = SectionIndex(articles, "ru")
print(f"статей {len(index.articles)}, разделов {len(index.sections)}")

In [ ]:
# Запросы с живой выдачей: (запрос, статьи живого топ-3, которые есть в корпусе).
titles = {a.title for a in index.articles}
topics = [json.loads(line) for line in (DATA / "topics_v2.jsonl").open(encoding="utf-8")]
queries = []
for record in topics:
    for query, hits in record["live_ru"].items():
        live = [h["title"] for h in hits[:3] if h["title"] in titles]
        if live:
            queries.append((query, live))
print(len(queries), "запросов")

def evaluate(weights):
    """Доля живого топ-3 в нашем топ-3 и доля совпавших первых мест."""
    overlap = first = 0.0
    for query, live in queries:
        ours = [h.article.title for h in index.search(query, k=3, weights=weights)]
        overlap += len(set(ours) & set(live)) / len(live)
        first += bool(ours) and ours[0] == live[0]
    return overlap / len(queries), first / len(queries)

print("без весов полей:", evaluate(Weights(title=0, heading=0, article=0)))
print("по умолчанию:", evaluate(Weights()))

In [ ]:
# Перебор весов (около 2 с на вариант).
results = []
for title, heading, article, k1, b in itertools.product([3, 10, 20, 40], [1, 3], [0, 0.1, 0.3, 1], [1.2, 2.0], [0.5, 0.75]):
    overlap, first = evaluate(Weights(k1=k1, b=b, title=title, heading=heading, article=article))
    results.append((round(overlap, 3), round(first, 3), dict(title=title, heading=heading, article=article, k1=k1, b=b)))
results.sort(key=lambda r: r[:2], reverse=True)
for r in results[:10]:
    print(r)

In [ ]:
# Промахи: запросы, где из живого топ-3 у нас нашлось меньше трети.
random.seed(1)
misses = []
for query, live in queries:
    ours = [h.article.title for h in index.search(query, k=3)]
    if len(set(ours) & set(live)) / len(live) < 0.34:
        misses.append((query, live, ours))
print(len(misses), "промахов")
for query, live, ours in random.sample(misses, 15):
    print(f"{query}\n  живой: {live}\n  наш:   {ours}")

In [ ]:
# Проверка по тезисам: содержит ли найденный раздел все слова термина (terms — запросы, кроме самих тем).
topic_texts = {r["topic"] for r in topics}
n = first_ok = any_ok = 0
for query, _ in queries:
    if query in topic_texts:
        continue
    stems = set(index.analyze(query))
    found = [set(index.analyze(h.article.title + " " + h.article.sections[h.section].heading + " "
                               + h.article.sections[h.section].text)) for h in index.search(query, k=3)]
    n += 1
    first_ok += bool(found) and stems <= found[0]
    any_ok += any(stems <= f for f in found)
print(f"terms: {n}; первый раздел содержит все слова — {first_ok / n:.2f}, хотя бы один из трёх — {any_ok / n:.2f}")